<a href="https://colab.research.google.com/github/EduardoAve/Labour-well-being/blob/main/notebooks/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import pandas as pd
import numpy as np
# Nota: KNNImputer y StandardScaler se importaron pero no se usaron en el
# código original proporcionado. Se mantienen aquí por si se usan más adelante.
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# --- ¡¡¡IMPORTANTE!!! REEMPLAZA ESTO CON LA RUTA A TU NUEVO ARCHIVO ---
file_path = '/content/drive/MyDrive/labour well being/Data prepatartion/Final_Dataset.xlsx'
# ---------------------------------------------------------------------

try:
    df = pd.read_excel(file_path)
    print(f"Archivo cargado exitosamente desde: {file_path}")
    print(f"Dimensiones iniciales del DataFrame: {df.shape}")
    # Verificar si 'Vulnerability' existe al inicio
    if 'Vulnerability' in df.columns:
        print("Columna 'Vulnerability' encontrada en el archivo original.")
    else:
        print("Advertencia: La columna 'Vulnerability' NO se encontró en el archivo original.")

except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta: {file_path}")
    exit() # Detener ejecución si el archivo no se encuentra
except Exception as e:
    print(f"Error al cargar el archivo Excel: {e}")
    exit()

# --- Bloque de renombrado ELIMINADO ---
# No es necesario ya que 'Vulnerability' existe con el nombre correcto.

# --- 1. Función de Inversión (Sin cambios) ---
def invert_scale(series):
    """Invierte valores en una escala de 1 a 5, ignorando NaNs."""
    if pd.api.types.is_numeric_dtype(series):
        return series.apply(lambda x: 6 - x if pd.notna(x) else x)
    else:
        return series

# --- 2. Cálculo de Variables (Bloque 1 - ÍNDICES AJUSTADOS +4 - Sin cambios) ---
print("\nCalculando variables: Academic Resources, Autonomy, Leadership, Community, Satisfaction, Burnout...")
# (Mismo código de cálculo que la versión anterior... Se omite por brevedad, pero debe estar aquí)
idx_academic_start = 36 + 4 # 40
idx_academic_end = 42 + 4   # 46
idx_pressure = 42 + 4       # 46
idx_autonomy_start = 43 + 4 # 47
idx_autonomy_end = 49 + 4   # 53 # El rango va hasta 52 (inclusive)
idx_autonomy_inv1 = 44 + 4  # 48
idx_autonomy_inv2 = 45 + 4  # 49
idx_autonomy_inv3 = 48 + 4  # 52
idx_leadership_start = 49 + 4 # 53
idx_leadership_end = 53 + 4   # 57
idx_community_start = 53 + 4  # 57
idx_community_end = 56 + 4    # 60
idx_satisfaction_start = 56 + 4 # 60
idx_satisfaction_end = 61 + 4   # 65
idx_burnout_start = 80 + 4    # 84
idx_burnout_end = 84 + 4      # 88
if idx_academic_end <= df.shape[1]: df['Academic_resources'] = df.iloc[:, idx_academic_start:idx_academic_end].mean(axis=1)
else: print(f"Advertencia: Rango de índice para Academic_resources ({idx_academic_start}:{idx_academic_end}) fuera de límites.")
if idx_pressure < df.shape[1]: df['Performance_pressure'] = df.iloc[:, idx_pressure]
else: print(f"Advertencia: Índice para Performance_pressure ({idx_pressure}) fuera de límites.")
if idx_autonomy_inv3 < df.shape[1]: # Chequea hasta el último índice necesario (52)
    df['Perceived_autonomy'] = (
        df.iloc[:, idx_autonomy_start] +                  # 47
        invert_scale(df.iloc[:, idx_autonomy_inv1]) +     # 48 inv
        invert_scale(df.iloc[:, idx_autonomy_inv2]) +     # 49 inv
        df.iloc[:, idx_autonomy_inv2 + 1] +               # 50
        df.iloc[:, idx_autonomy_inv2 + 2] +               # 51
        invert_scale(df.iloc[:, idx_autonomy_inv3])       # 52 inv
    ) / 6
else: print(f"Advertencia: Rango de índice para Perceived_autonomy (hasta {idx_autonomy_inv3}) fuera de límites.")
if idx_leadership_end <= df.shape[1]: df['Quality_of_leadership'] = df.iloc[:, idx_leadership_start:idx_leadership_end].mean(axis=1)
else: print(f"Advertencia: Rango de índice para Quality_of_leadership ({idx_leadership_start}:{idx_leadership_end}) fuera de límites.")
if idx_community_end <= df.shape[1]: df['Sense_of_community'] = df.iloc[:, idx_community_start:idx_community_end].mean(axis=1)
else: print(f"Advertencia: Rango de índice para Sense_of_community ({idx_community_start}:{idx_community_end}) fuera de límites.")
if idx_satisfaction_end <= df.shape[1]: df['Job_satisfaction'] = df.iloc[:, idx_satisfaction_start:idx_satisfaction_end].mean(axis=1)
else: print(f"Advertencia: Rango de índice para Job_satisfaction ({idx_satisfaction_start}:{idx_satisfaction_end}) fuera de límites.")
if idx_burnout_end <= df.shape[1]: df['Burnout'] = df.iloc[:, idx_burnout_start:idx_burnout_end].mean(axis=1)
else: print(f"Advertencia: Rango de índice para Burnout ({idx_burnout_start}:{idx_burnout_end}) fuera de límites.")
print("Cálculo Bloque 1 completado.")


# --- 3. Cálculo de Variables de Motivación (Bloque 2 - Sin cambios) ---
print("\nCalculando variables de Motivación Laboral (con índices ajustados)...")
# (Mismo código de cálculo que la versión anterior... Se omite por brevedad, pero debe estar aquí)
amotivation_indices = [i + 4 for i in [61, 67, 73]]
extrinsic_social_indices = [i + 4 for i in [62, 68, 74]]
extrinsic_material_indices = [i + 4 for i in [63, 69, 75]]
introjected_indices = [i + 4 for i in [64, 70, 76, 79]]
identified_indices = [i + 4 for i in [65, 71, 77]]
intrinsic_indices = [i + 4 for i in [66, 72, 78]]
def calculate_mean_if_valid(df, indices, new_col_name):
    if not indices:
         print(f"Advertencia: Lista de índices vacía para '{new_col_name}'.")
         return
    max_index = max(indices)
    if max_index < df.shape[1]:
        valid_indices = [idx for idx in indices if idx < df.shape[1]]
        if len(valid_indices) == len(indices):
             df[new_col_name] = df.iloc[:, valid_indices].mean(axis=1)
        else:
             print(f"Advertencia: No todos los índices para '{new_col_name}' son válidos. Se omitirá el cálculo.")
    else:
        print(f"Advertencia: No se pudo calcular '{new_col_name}' - índice máximo requerido {max_index} está fuera de rango (total cols: {df.shape[1]}).")
calculate_mean_if_valid(df, amotivation_indices, 'Amotivation')
calculate_mean_if_valid(df, extrinsic_social_indices, 'Extrinsic_Social')
calculate_mean_if_valid(df, extrinsic_material_indices, 'Extrinsic_Material')
calculate_mean_if_valid(df, introjected_indices, 'Introjected')
calculate_mean_if_valid(df, identified_indices, 'Identified')
calculate_mean_if_valid(df, intrinsic_indices, 'Intrinsic')
print("Cálculo Bloque 2 (Motivación) completado.")


# --- 4. Cálculo de Indicadores AVEM-44 (Bloque 3 - Sin cambios) ---
print("\nCalculando indicadores AVEM-44 (Vulnerabilidad, con índice base ajustado)...")
# (Mismo código de cálculo que la versión anterior... Se omite por brevedad, pero debe estar aquí)
base_avem_index = 84 + 4 # Nuevo índice base = 88
items_to_reverse = [5, 19, 22, 27, 33, 38] # Números relativos
reversed_cols = {}
avem_scales = {
    'Subjective_significance_of_work': [1, 12, 23, 34], 'Professional_ambition': [2, 13, 24, 35],
    'Tendency_to_exert': [3, 14, 25, 36], 'Striving_for_perfection': [4, 15, 26, 37],
    'Emotional_distancing': [5, 16, 27, 38], 'Resignation_tendencies': [6, 17, 28, 39],
    'Offensive_coping_with_problems': [7, 18, 29, 40], 'Balance_and_mental_stability': [8, 19, 30, 41],
    'Satisfaction_with_work': [9, 20, 31, 42], 'Satisfaction_with_life': [10, 21, 32, 43],
    'Experience_of_social_support': [11, 22, 33, 44]
}
all_item_nums = [item for sublist in avem_scales.values() for item in sublist]
max_item_num_overall = max(all_item_nums) if all_item_nums else 0
max_overall_avem_idx = base_avem_index + max_item_num_overall - 1
if max_overall_avem_idx < df.shape[1]:
    possible_to_reverse = True
    max_item_num_to_reverse = max(items_to_reverse) if items_to_reverse else 0
    max_idx_to_reverse = base_avem_index + max_item_num_to_reverse - 1
    if max_idx_to_reverse < df.shape[1]:
         for item_num in items_to_reverse:
             col_index = base_avem_index + item_num - 1
             reversed_series = invert_scale(df.iloc[:, col_index])
             reversed_cols[item_num] = reversed_series
    else:
        print(f"Advertencia: Índice máximo para invertir items AVEM ({max_idx_to_reverse}) fuera de rango. No se invertirán.")
        possible_to_reverse = False
    for scale_name, item_list in avem_scales.items():
        scale_data = pd.DataFrame(index=df.index)
        valid_items_count = 0
        for item_num in item_list:
            col_index = base_avem_index + item_num - 1
            if item_num in items_to_reverse and possible_to_reverse and item_num in reversed_cols:
                 scale_data[f'item_{item_num}'] = reversed_cols[item_num]
            else:
                 scale_data[f'item_{item_num}'] = df.iloc[:, col_index]
            valid_items_count += 1
        if valid_items_count > 0:
            df[scale_name] = scale_data.mean(axis=1)
        else:
            print(f"Error: No se encontraron columnas válidas para calcular '{scale_name}'.")
else:
     print(f"Advertencia: Índice máximo para calcular escalas AVEM ({max_overall_avem_idx}) fuera de rango. No se calcularán.")
print("Cálculo Bloque 3 (AVEM-44) completado.")


# --- 5. Seleccionar Columnas Finales (Lógica simplificada para 'Vulnerability') ---
print("\n--- PASO 5: Seleccionando columnas finales a mantener ---")

# Define columnas iniciales a mantener: indices 0 a 38 inclusive. Rango es 0 a 39.
initial_cols_indices = list(range(0, 39))
initial_cols_names = []

# Verifica que el índice máximo necesario exista
if initial_cols_indices and (max(initial_cols_indices) < df.shape[1]):
     initial_cols_names = df.columns[initial_cols_indices].tolist()
     print(f"Se conservarán las primeras {len(initial_cols_names)} columnas iniciales (índices {initial_cols_indices[0]}-{initial_cols_indices[-1]}).")
else:
    print(f"Advertencia: Rango de índices iniciales {initial_cols_indices} inválido para las {df.shape[1]} columnas actuales. No se incluirán columnas iniciales.")

# Nombres de las columnas que ESTE SCRIPT calcula
calculated_col_names = [
    'Academic_resources', 'Performance_pressure', 'Perceived_autonomy',
    'Quality_of_leadership', 'Sense_of_community', 'Job_satisfaction', 'Burnout',
    'Amotivation', 'Extrinsic_Social', 'Extrinsic_Material',
    'Introjected', 'Identified', 'Intrinsic',
    'Subjective_significance_of_work', 'Professional_ambition', 'Tendency_to_exert',
    'Striving_for_perfection', 'Emotional_distancing', 'Resignation_tendencies',
    'Offensive_coping_with_problems', 'Balance_and_mental_stability',
    'Satisfaction_with_work', 'Satisfaction_with_life', 'Experience_of_social_support'
]
# Filtrar por las que realmente existen en el df después del cálculo
calculated_cols_existing = [col for col in calculated_col_names if col in df.columns]
if len(calculated_cols_existing) < len(calculated_col_names):
    print("Advertencia: Algunas columnas calculadas no existen en el DataFrame y no se incluirán.")
    missing_calculated = set(calculated_col_names) - set(calculated_cols_existing)
    print(f"Columnas calculadas faltantes: {list(missing_calculated)}")

# Nombre de la columna de vulnerabilidad que ya existe
vulnerability_col_name = 'Vulnerability'

# Construir la lista final: Iniciales + Calculadas + Vulnerability (si existe y no está ya)
final_col_list = initial_cols_names + calculated_cols_existing

# Añadir 'Vulnerability' al final SI existe en el df y NO está ya incluida en las anteriores
if vulnerability_col_name in df.columns:
    if vulnerability_col_name not in final_col_list:
        final_col_list.append(vulnerability_col_name)
        print(f"Se incluirá la columna existente '{vulnerability_col_name}' al final.")
    # else: # No imprimir nada si ya estaba (no debería pasar con indices 0-38)
    #    pass
else:
    print(f"Advertencia: La columna '{vulnerability_col_name}' no se encontró en el DataFrame al momento de la selección final.")


# Crear el nuevo DataFrame manteniendo solo estas columnas y usando .copy()
if final_col_list:
    # Doble chequeo para asegurar que todas las columnas existen en df ANTES de seleccionar
    final_col_list_verified = [col for col in final_col_list if col in df.columns]
    if len(final_col_list_verified) != len(final_col_list):
        print("Advertencia: Algunas columnas en la lista final no existen en el DataFrame. Se usarán solo las existentes.")
        missing_final = set(final_col_list) - set(df.columns)
        print(f"Columnas omitidas de la selección final: {list(missing_final)}")

    print(f"\nSe conservarán un total de {len(final_col_list_verified)} columnas en el orden especificado.")
    # Asegurarse de no tener duplicados al final (aunque la lógica previene esto)
    seen = set()
    final_col_list_unique = [x for x in final_col_list_verified if not (x in seen or seen.add(x))]

    df_final = df[final_col_list_unique].copy()

    # --- Verificación Final ---
    print("\nDataFrame después de calcular variables y seleccionar columnas finales:")
    print(f"Dimensiones finales del DataFrame: {df_final.shape}")
    print("Nombres de las columnas finales:")
    print(df_final.columns.tolist())

    # --- Opcional: Guardar el resultado ---
    # output_file_path = 'ruta/donde/guardar/resultado_preparado.xlsx'
    # print(f"\nGuardando DataFrame final en: {output_file_path}")
    # df_final.to_excel(output_file_path, index=False)
    # print("Guardado completado.")

else:
    print("\nError: No se pudieron determinar las columnas a mantener. No se generó el DataFrame final.")

Archivo cargado exitosamente desde: /content/drive/MyDrive/labour well being/Data prepatartion/Final_Dataset.xlsx
Dimensiones iniciales del DataFrame: (2747, 132)
Columna 'Vulnerability' encontrada en el archivo original.

Calculando variables: Academic Resources, Autonomy, Leadership, Community, Satisfaction, Burnout...
Cálculo Bloque 1 completado.

Calculando variables de Motivación Laboral (con índices ajustados)...
Cálculo Bloque 2 (Motivación) completado.

Calculando indicadores AVEM-44 (Vulnerabilidad, con índice base ajustado)...
Cálculo Bloque 3 (AVEM-44) completado.

--- PASO 5: Seleccionando columnas finales a mantener ---
Se conservarán las primeras 39 columnas iniciales (índices 0-38).
Se incluirá la columna existente 'Vulnerability' al final.

Se conservarán un total de 64 columnas en el orden especificado.

DataFrame después de calcular variables y seleccionar columnas finales:
Dimensiones finales del DataFrame: (2747, 64)
Nombres de las columnas finales:
['Country', 'Ver

In [58]:
# --- Bloque 2: Renombrado y Limpieza Inicial (Pasos 6-10 en la numeración anterior) ---

# Asegúrate de que df_final exista y sea el resultado del Bloque 1
if 'df_final' not in locals() or df_final is None:
    print("ERROR: El DataFrame 'df_final' no está definido. Cárgalo o créalo primero.")
    # exit() # O maneja el error como prefieras
else:
    print(f"\nIniciando Bloque 2 sobre df_final con shape: {df_final.shape}")

    # --- 1. Eliminaciones Iniciales (Usando nombres originales de df_final) ---
    print("\n--- PASO 6: Eliminando columnas iniciales no necesarias ---")
    cols_to_drop_initial = [
        # La primera columna '14.1...' (identificada previamente en índice 21)
        '14.1. Actual average weekly working hours outside higher education (in a typical semester week): ',
        # La columna 'Unnamed: 4'
        'Unnamed: 4',
        # La columna 'Nationality'
        '3. Nationality:',
        # *** ELIMINAR Income CZK ***
        'Income CZK'
        # ****************************
    ]
    # Verificar si las columnas existen antes de intentar eliminarlas
    existing_cols_to_drop = [col for col in cols_to_drop_initial if col in df_final.columns]
    if existing_cols_to_drop:
        # Usar errors='ignore' por si alguna columna ya fue eliminada o no existe
        df_final.drop(columns=existing_cols_to_drop, inplace=True, errors='ignore')
        print(f"Columnas eliminadas inicialmente: {existing_cols_to_drop}")
        print(f"Dimensiones después de eliminación inicial: {df_final.shape}")
    else:
        print("No se encontraron columnas iniciales para eliminar (o ya fueron eliminadas).")


    # --- 2. Renombrado de Columnas (PASO 7) ---
    print("\n--- PASO 7: Renombrando columnas ---")
    # Asegúrate de tener aquí tu diccionario 'rename_dict_refined' completo y actualizado
    # (el que usamos antes, pero la entrada para "Income CZK" ya no es necesaria)
    rename_dict_refined = {
        # Información básica y demográfica
        "1. Gender:": "Gender", "2. Age (in years):": "Age",
        "4. Current marital (partnership) status:": "MaritalStatus",
        "5. Do you currently care for underage children or dependent relatives?": "CareResponsibilities",
        # Info Institucional
        "6. The type of higher education insitution where you primarily work:": "HeiType",
        "7. Subject area of the faculty (higher education institution) where you primarily work:": "FacultySubjectArea",
        "8. Duration of your current employment contract at the higher education institution where you primarily work:": "EmploymentContractDuration",
        "9. Extent of employment in higher education (in hours/week, aggregated for all higher education institutions where you work):": "HeiEmploymentHours",
        "10. Actual average weekly working hours in higher education (in a typical semester week):": "HeiActualWeeklyHours",
        "Effort (less, more, equal)": "EffortLevel", "Effort [%]": "EffortPercentage",
        # Ingresos restantes
        # "Income CZK": "IncomeCZK", # Ya no existe
        "Income EURO": "IncomeEURO", "Euro Adj.": "EuroAdjusted",
        "Salary/hour": "SalaryPerHour", "Salary effort/hour": "SalaryEffortPerHour",
        # Posición y Trabajo
        "12. Do you hold a leadership position at a higher education institution?": "LeadershipPosition",
        "13. How influential are you in helping to shape key academic policies at your institution at the level of department or similar unit?": "PolicyInfluence",
        "14. Do you currently have another (paid) job outside higher education?": "OtherPaidJob",
        '14.1. Actual average weekly working hours outside higher education (in a typical semester week): .1': 'OtherJobWeeklyHours1', # Renombrar la '.1'
        "Academic/Non-academic": "AcademicOrNonAcademic",
        "CZ_15. Your current position at the higher education institution, where you primarily work: ": "CurrentPositionCZ",
        "AT_15. Your current position at the higher education institution, where you primarily work: ": "CurrentPositionAT",
        "16. Please choose the category that best fits your job description:": "JobCategory",
        "17. The highest level of education attained:": "HighestEducationLevel",
        "18. Total length of your career in Czech higher education in years:": "CareerLengthCZ",
        # Descripciones Actividad
        "1. Teaching (classroom instruction, preparation of instructional materials and lesson plans, advising students, reading and evaluating student work, examination management, etc.)": "TeachingActivityDesc",
        "2. Research (reading literature, designing and conducting experiments, collecting and analysing data, writing articles or other scientific texts, etc.)": "ResearchActivityDesc",
        "3. Activities related to externally funded research projects (searching for information on available funding sources, preparation of grant applications and project reports, project management and administration, etc.)": "FundedResearchActivityDesc",
        "4. Organisational and administrative activities (organising and attending meetings, dealing with tasks and documents not directly related to teaching, research, or externally funded research projects, etc.)": "AdminActivityDesc",
        # Porcentajes Actividad
        "Teaching %": "TeachingPercentage", "Research %": "ResearchPercentage",
        "Activities related to externally funded research projects %": "FundedResearchPercentage",
        "Organisational and administrative activities %": "AdminPercentage",
        # Columnas Calculadas (CamelCase)
        "Academic_resources": "AcademicResources", "Performance_pressure": "PerformancePressure",
        "Perceived_autonomy": "PerceivedAutonomy", "Quality_of_leadership": "QualityOfLeadership",
        "Sense_of_community": "SenseOfCommunity", "Job_satisfaction": "JobSatisfaction",
        "Burnout": "Burnout", "Amotivation": "Amotivation", "Extrinsic_Social": "ExtrinsicSocial",
        "Extrinsic_Material": "ExtrinsicMaterial", "Introjected": "Introjected",
        "Identified": "Identified", "Intrinsic": "Intrinsic",
        "Subjective_significance_of_work": "SubjectiveSignificanceOfWork",
        "Professional_ambition": "ProfessionalAmbition", "Tendency_to_exert": "TendencyToExert",
        "Striving_for_perfection": "StrivingForPerfection", "Emotional_distancing": "EmotionalDistancing",
        "Resignation_tendencies": "ResignationTendencies", "Offensive_coping_with_problems": "OffensiveCopingWithProblems",
        "Balance_and_mental_stability": "BalanceAndMentalStability", "Satisfaction_with_work": "SatisfactionWithWork",
        "Satisfaction_with_life": "SatisfactionWithLife", "Experience_of_social_support": "ExperienceOfSocialSupport",
        # Vulnerability ya está bien nombrada
    }
    rename_dict_applicable = {k: v for k, v in rename_dict_refined.items() if k in df_final.columns}
    df_final = df_final.rename(columns=rename_dict_applicable)
    print(f"Se renombraron {len(rename_dict_applicable)} columnas.")


    # --- 3. Conversión de Tipos (PASO 8 - Usando nombres NUEVOS) ---
    print("\n--- PASO 8: Convirtiendo columnas a tipo numérico ---")
    # ¡Revisa si 'AdminActivityDesc' es la correcta o si era 'AdminPercentage'!
    cols_to_numeric = ['HeiEmploymentHours', 'CareerLengthCZ', 'AdminActivityDesc']
    for col in cols_to_numeric:
        if col in df_final.columns:
            original_dtype = df_final[col].dtype
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
            nan_count = df_final[col].isnull().sum()
            print(f"Columna '{col}' convertida a numérica. Tipo original: {original_dtype}. ({nan_count} NaNs generados/existentes)")
        else:
            print(f"Advertencia: Columna '{col}' no encontrada para conversión numérica.")


    # --- 4. Combinación y Eliminación de Columnas de Posición (PASO 9 - Usando nombres NUEVOS) ---
    print("\n--- PASO 9: Combinando CurrentPosition y eliminando originales ---")
    cz_col = 'CurrentPositionCZ'; at_col = 'CurrentPositionAT'; new_col = 'CurrentPosition'
    if cz_col in df_final.columns and at_col in df_final.columns:
        print(f"Combinando '{cz_col}' y '{at_col}' en '{new_col}' usando suma (skipna=True).")
        # Considera combine_first si solo una debe tener valor: df_final[new_col] = df_final[cz_col].combine_first(df_final[at_col])
        df_final[new_col] = df_final[[cz_col, at_col]].sum(axis=1, skipna=True)
        df_final.drop(columns=[cz_col, at_col], inplace=True)
        print(f"Columnas '{cz_col}' y '{at_col}' eliminadas.")
    elif cz_col in df_final.columns:
         print(f"Advertencia: Solo se encontró '{cz_col}', se renombrará a '{new_col}'.")
         df_final.rename(columns={cz_col: new_col}, inplace=True)
    elif at_col in df_final.columns:
         print(f"Advertencia: Solo se encontró '{at_col}', se renombrará a '{new_col}'.")
         df_final.rename(columns={at_col: new_col}, inplace=True)
    else:
        print(f"Advertencia: No se encontraron las columnas '{cz_col}' ni '{at_col}' para combinar.")
    print(f"Dimensiones después de combinar/eliminar posición: {df_final.shape}")


    # --- 5. Operaciones a Nivel de Fila (PASO 10) ---
    print("\n--- PASO 10: Eliminando duplicados y rellenando NaNs ---")
    # Eliminar filas duplicadas
    initial_rows = df_final.shape[0]
    df_final = df_final.drop_duplicates(keep='first')
    rows_dropped = initial_rows - df_final.shape[0]
    print(f"Se eliminaron {rows_dropped} filas duplicadas. Filas restantes: {df_final.shape[0]}")
    # Rellenar NaNs en 'OtherPaidJob'
    job_col = 'OtherPaidJob'
    if job_col in df_final.columns:
        nan_before = df_final[job_col].isnull().sum()
        fill_value = 1 # ¡Confirma si este valor es correcto!
        print(f"Rellenando {nan_before} NaNs en '{job_col}' con el valor '{fill_value}'.")
        df_final[job_col] = df_final[job_col].fillna(fill_value)
    else:
        print(f"Advertencia: Columna '{job_col}' no encontrada para rellenar NaNs.")


    # --- Fin del Bloque 2 ---
    print("\n--- Fin del Bloque de Renombrado y Limpieza Inicial ---")
    print(f"Dimensiones finales de df_final (listo para Pasos 1-4): {df_final.shape}")

# Ahora df_final está listo para entrar al Bloque 3 (Pasos 1-4 de limpieza NaN/Outlier)
# y ya no contiene la columna IncomeCZK.


Iniciando Bloque 2 sobre df_final con shape: (2747, 64)

--- PASO 6: Eliminando columnas iniciales no necesarias ---
Columnas eliminadas inicialmente: ['14.1. Actual average weekly working hours outside higher education (in a typical semester week): ', 'Unnamed: 4', '3. Nationality:', 'Income CZK']
Dimensiones después de eliminación inicial: (2747, 60)

--- PASO 7: Renombrando columnas ---
Se renombraron 57 columnas.

--- PASO 8: Convirtiendo columnas a tipo numérico ---
Columna 'HeiEmploymentHours' convertida a numérica. Tipo original: object. (37 NaNs generados/existentes)
Columna 'CareerLengthCZ' convertida a numérica. Tipo original: object. (2207 NaNs generados/existentes)
Columna 'AdminActivityDesc' convertida a numérica. Tipo original: object. (667 NaNs generados/existentes)

--- PASO 9: Combinando CurrentPosition y eliminando originales ---
Combinando 'CurrentPositionCZ' y 'CurrentPositionAT' en 'CurrentPosition' usando suma (skipna=True).
Columnas 'CurrentPositionCZ' y 'Curren

In [59]:
for col in df_final.columns:
    print(f"Columna: {col}, Tipo: {df_final[col].dtype}")

Columna: Country, Tipo: int64
Columna: Version, Tipo: int64
Columna: Gender, Tipo: float64
Columna: Age, Tipo: float64
Columna: MaritalStatus, Tipo: float64
Columna: CareResponsibilities, Tipo: float64
Columna: HeiType, Tipo: float64
Columna: FacultySubjectArea, Tipo: float64
Columna: EmploymentContractDuration, Tipo: float64
Columna: HeiEmploymentHours, Tipo: float64
Columna: HeiActualWeeklyHours, Tipo: float64
Columna: EffortLevel, Tipo: float64
Columna: EffortPercentage, Tipo: float64
Columna: IncomeEURO, Tipo: float64
Columna: EuroAdjusted, Tipo: float64
Columna: SalaryPerHour, Tipo: object
Columna: SalaryEffortPerHour, Tipo: float64
Columna: LeadershipPosition, Tipo: float64
Columna: PolicyInfluence, Tipo: float64
Columna: OtherPaidJob, Tipo: float64
Columna: OtherJobWeeklyHours1, Tipo: float64
Columna: AcademicOrNonAcademic, Tipo: int64
Columna: JobCategory, Tipo: float64
Columna: HighestEducationLevel, Tipo: float64
Columna: CareerLengthCZ, Tipo: float64
Columna: TeachingActivit

In [60]:
# --- Bloque 3: Limpieza NaNs/Outliers (Pasos 1-4) ---

# Asumiendo que 'df_final' es el DataFrame resultante del Bloque 2
# (después de renombrar, eliminar IncomeCZK, etc.)
if 'df_final' not in locals() or df_final is None:
    print("ERROR: El DataFrame 'df_final' (salida del Bloque 2) no está definido.")
    # exit() # O maneja el error
else:
    print(f"\nIniciando Bloque 3 (Limpieza NaNs/Outliers) sobre df_final con shape: {df_final.shape}")

    # --- Configuración General ---
    # Re-definir columnas estructurales por si acaso (con nombres finales)
    structural_nan_cols = [
        'JobCategory', 'HighestEducationLevel', 'CareerLengthCZ',
        'TeachingActivityDesc', 'ResearchActivityDesc', 'FundedResearchActivityDesc',
        'AdminActivityDesc'
    ]
    structural_nan_cols = [col for col in structural_nan_cols if col in df_final.columns]
    print(f"\nColumnas estructurales con NaNs siempre permitidos: {structural_nan_cols}")

    # ==============================================================================
    # PASO 1: Filtrar Filas por Límite de NaNs (Umbral=6, excluyendo estructurales)
    # ==============================================================================
    print("\n--- PASO 1: Filtrando filas con > 6 NaNs (no estructurales) ---")
    max_allowed_nans = 6 # <-- Umbral 6
    all_cols_step1 = df_final.columns.tolist()
    cols_to_check_nans_step1 = [col for col in all_cols_step1 if col not in structural_nan_cols]
    nan_counts_per_row_step1 = df_final[cols_to_check_nans_step1].isnull().sum(axis=1)
    rows_to_keep_mask_step1 = nan_counts_per_row_step1 <= max_allowed_nans
    rows_before_step1 = len(df_final)
    # Crear df_step1 como resultado de este paso
    df_step1 = df_final[rows_to_keep_mask_step1].copy()
    rows_after_step1 = len(df_step1)
    print(f"Filas antes: {rows_before_step1}")
    print(f"Filas después: {rows_after_step1}")
    print(f"Filas eliminadas en Paso 1: {rows_before_step1 - rows_after_step1}")

    # ==============================================================================
    # PASO 2: Eliminar Outliers de Ingresos con IQR (Sin cambios)
    # ==============================================================================
    print("\n--- PASO 2: Eliminando outliers de Ingresos (IQR) ---")
    cols_for_iqr = ['IncomeEURO', 'EuroAdjusted'] # IncomeCZK ya no existe
    cols_for_iqr = [col for col in cols_for_iqr if col in df_step1.columns]
    if cols_for_iqr:
        rows_to_keep_mask_step2 = pd.Series(True, index=df_step1.index)
        for col in cols_for_iqr:
            if pd.api.types.is_numeric_dtype(df_step1[col]):
                Q1 = df_step1[col].quantile(0.25); Q3 = df_step1[col].quantile(0.75); IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR; upper_bound = Q3 + 1.5 * IQR
                print(f"  Límites IQR para {col}: Inferior={lower_bound:.2f}, Superior={upper_bound:.2f}")
                outliers_mask_col = ((df_step1[col] < lower_bound) | (df_step1[col] > upper_bound)).fillna(False)
                print(f"  Outliers detectados en {col}: {outliers_mask_col.sum()}")
                rows_to_keep_mask_step2 = rows_to_keep_mask_step2 & (~outliers_mask_col)
            else: print(f"  Advertencia: Columna '{col}' no es numérica, se omite para IQR.")
        rows_before_step2 = len(df_step1)
        # Crear df_step2 como resultado de este paso
        df_step2 = df_step1[rows_to_keep_mask_step2].copy()
        rows_after_step2 = len(df_step2)
        print(f"Filas antes: {rows_before_step2}")
        print(f"Filas después: {rows_after_step2}")
        print(f"Filas eliminadas en Paso 2: {rows_before_step2 - rows_after_step2}")
    else:
        print("  No se encontraron columnas de ingresos especificadas para IQR. Omitiendo paso.")
        df_step2 = df_step1.copy() # Pasar al siguiente paso


    # ==============================================================================
    # PASO 3: Imputación de Valores Faltantes con KNN (Sin cambios en la lógica interna)
    # ==============================================================================
    print("\n--- PASO 3: Imputando NaNs restantes con KNN (k=5) ---")
    # Lista de columnas objetivo (donde nos gustaría que se imputaran NaNs)
    cols_to_impute_step3 = [
        'Age', 'HeiActualWeeklyHours', 'EffortLevel', 'EffortPercentage',
        'IncomeEURO', 'EuroAdjusted', 'SalaryEffortPerHour',
        'AcademicResources', 'PerceivedAutonomy',
        'TeachingPercentage', 'ResearchPercentage', 'FundedResearchPercentage', 'AdminPercentage'
    ]
    cols_to_impute_step3 = [col for col in cols_to_impute_step3 if col in df_step2.columns]
    print(f"Columnas objetivo cuya imputación se verificará explícitamente: {cols_to_impute_step3}")

    # Columnas a EXCLUIR del proceso KNN (IncomeCZK ya no está)
    cols_to_exclude_from_matrix_step3 = list(set(structural_nan_cols + [
        'Country', 'Version', 'Gender', 'MaritalStatus', 'CareResponsibilities',
        'HeiType', 'FacultySubjectArea', 'EmploymentContractDuration',
        'LeadershipPosition', 'PolicyInfluence', 'OtherPaidJob',
        'OtherJobWeeklyHours1', 'AcademicOrNonAcademic', 'CurrentPosition',
        'IncomeEUR', 'SalaryPerHour' # Excluida por ser object
    ]))
    cols_to_exclude_from_matrix_step3 = [col for col in cols_to_exclude_from_matrix_step3 if col in df_step2.columns]

    numeric_cols_step3 = df_step2.select_dtypes(include=np.number).columns.tolist()
    numeric_cols_for_imputation_step3 = [
        col for col in numeric_cols_step3
        if col not in cols_to_exclude_from_matrix_step3
    ]
    print(f"Se usarán {len(numeric_cols_for_imputation_step3)} columnas numéricas como base para KNN.")

    # Crear df_step3 para guardar el resultado (o no) de KNN
    df_step3 = df_step2.copy()

    if not numeric_cols_for_imputation_step3:
        print("¡Advertencia! No se encontraron columnas numéricas adecuadas para KNN. Se omite la imputación.")
    else:
        df_numeric_subset = df_step3[numeric_cols_for_imputation_step3]
        numeric_subset_cols = df_numeric_subset.columns
        numeric_subset_index = df_numeric_subset.index
        nans_in_subset = df_numeric_subset.isnull().sum().sum()
        print(f"  NaNs encontrados en el subset numérico para KNN: {nans_in_subset}")

        if nans_in_subset > 0:
            try:
                # --- Inicio Código KNN ---
                # (Asegúrate de tener aquí tu código completo y funcional para escalar,
                #  instanciar KNNImputer, hacer fit_transform, invertir escala
                #  y actualizar df_step3 con los resultados de df_imputed_subset)
                # Ejemplo de estructura mínima:
                print("  Escalando datos...")
                scaler = StandardScaler()
                # ... (manejo de varianza cero si es necesario) ...
                scaled_data = scaler.fit_transform(df_numeric_subset)
                print("  Aplicando KNNImputer...")
                imputer = KNNImputer(n_neighbors=5)
                imputed_scaled_data = imputer.fit_transform(scaled_data)
                print("  Invirtiendo escalado...")
                imputed_original_scale_data = scaler.inverse_transform(imputed_scaled_data)
                df_imputed_subset = pd.DataFrame(imputed_original_scale_data, columns=numeric_subset_cols, index=numeric_subset_index)
                print("  Actualizando DataFrame principal (df_step3)...")
                update_count = 0
                for col in numeric_subset_cols:
                    if col in df_imputed_subset.columns:
                         if df_step3[col].isnull().any():
                              df_step3[col] = df_imputed_subset[col]
                              update_count += 1
                print(f"  Se actualizaron datos imputados para {update_count} columnas.")
                # --- Fin Código KNN ---
            except Exception as e:
                print(f"  ERROR durante la imputación KNN: {e}. Omitiendo imputación.")
                # df_step3 sigue siendo la copia de df_step2
        else:
            print("  No se encontraron NaNs en las columnas seleccionadas para KNN, se omite la imputación.")

    # --- Diagnóstico Opcional Pre-Step 4 ---
    # print("\n--- Verificación Pre-Step 4: NaNs en columnas con > 0 NaNs ---")
    # nans_before_step4_detailed = df_step3.isnull().sum()
    # print(nans_before_step4_detailed[nans_before_step4_detailed > 0])
    # print("--------------------------------------------------------")


    # ==============================================================================
    # PASO 4: Eliminar Filas con NaNs Restantes (IGNORANDO COLUMNAS PROBLEMÁTICAS) <-- LÓGICA CLAVE
    # ==============================================================================
    print("\n--- PASO 4: Eliminando filas con NaNs restantes (Ignorando columnas problemáticas) ---")

    # Define columnas donde los NaNs son aceptables PARA ESTE dropna específico
    # (IncomeCZK ya no existe aquí)
    cols_to_ignore_nans_step4 = structural_nan_cols + [
        'TeachingPercentage', 'ResearchPercentage', 'FundedResearchPercentage', 'AdminPercentage', # Las 4 de %
        'SalaryPerHour', # Recordar que es tipo 'object'
        'Vulnerability',
        # Añadir estas también ya que KNN pudo no haberlas limpiado completamente
        'IncomeEURO',
        'EuroAdjusted',
        'SalaryEffortPerHour',
        # Añadir otras que vimos con NaNs antes y que no son críticas para eliminar filas
        'Age', 'PerceivedAutonomy', 'Gender', 'MaritalStatus', 'CareResponsibilities',
        'HeiType', 'FacultySubjectArea', 'EmploymentContractDuration', 'HeiEmploymentHours',
        'EffortLevel', 'LeadershipPosition', 'PolicyInfluence', 'OtherPaidJob',
        'OtherJobWeeklyHours1', 'AcademicOrNonAcademic', 'CurrentPosition'
        # Prácticamente todas las que no son las escalas calculadas principales
    ]
    cols_to_ignore_nans_step4 = sorted(list(set([col for col in cols_to_ignore_nans_step4 if col in df_step3.columns])))
    print(f"Se IGNORARÁN los NaNs en las siguientes columnas para dropna ({len(cols_to_ignore_nans_step4)}): {cols_to_ignore_nans_step4}")

    # Define el subconjunto de columnas a REVISAR para dropna: TODAS MENOS las ignoradas
    all_cols_step4 = df_step3.columns.tolist()
    cols_to_check_for_nans_step4 = [col for col in all_cols_step4 if col not in cols_to_ignore_nans_step4]

    print(f"Se REVISARÁN {len(cols_to_check_for_nans_step4)} columnas para eliminar filas con NaNs.")
    if cols_to_check_for_nans_step4:
        nans_before_dropna = df_step3[cols_to_check_for_nans_step4].isnull().sum().sum()
        print(f"NaNs totales en columnas a REVISAR ANTES de dropna: {nans_before_dropna}")
    else:
        nans_before_dropna = 0
        print("No hay columnas especificadas para revisar NaNs en este paso.")

    rows_before_step4 = len(df_step3)
    # Aplicar dropna sobre el subconjunto de columnas REVISADAS
    if cols_to_check_for_nans_step4 and nans_before_dropna > 0 :
        # El resultado es el DataFrame final limpio (para este bloque)
        df_final_cleaned = df_step3.dropna(subset=cols_to_check_for_nans_step4)
    else:
        df_final_cleaned = df_step3.copy()
        print("No se eliminaron filas por NaNs en este paso (columnas ignoradas o sin NaNs).")

    rows_after_step4 = len(df_final_cleaned)
    print(f"Filas antes: {rows_before_step4}")
    print(f"Filas después: {rows_after_step4}")
    print(f"Filas eliminadas en Paso 4: {rows_before_step4 - rows_after_step4}")

    # --- Verificación Final del Paso 4 ---
    if cols_to_check_for_nans_step4:
        nan_check_sum_step4 = df_final_cleaned[cols_to_check_for_nans_step4].isnull().sum().sum()
        print(f"\nSuma total de NaNs en columnas REVISADAS después del dropna final: {nan_check_sum_step4}")
        if nan_check_sum_step4 != 0:
            print("¡¡¡ADVERTENCIA!!! Todavía quedan NaNs inesperados en columnas que debían limpiarse.")
        else:
            print("Verificación completada: No quedan NaNs en columnas revisadas.")
    else:
        print("\nNo se revisaron columnas para NaNs en este paso.")

    print("\nNaNs restantes por columna (deberían quedar en estructurales e ignoradas):")
    nan_summary = df_final_cleaned.isnull().sum()
    print(nan_summary[nan_summary > 0]) # Mostrar solo columnas con NaNs

    # --- Resultado de este bloque ---
    print("\n==============================================================================")
    print("PROCESO DE LIMPIEZA (PASOS 1-4) COMPLETADO")
    print(f"El DataFrame 'df_final_cleaned' (entrada para Paso 5) tiene: {df_final_cleaned.shape[0]} filas y {df_final_cleaned.shape[1]} columnas.")
    print("==============================================================================")

# Ahora puedes ejecutar el código del Paso 5 usando df_final_cleaned como entrada.


Iniciando Bloque 3 (Limpieza NaNs/Outliers) sobre df_final con shape: (2747, 59)

Columnas estructurales con NaNs siempre permitidos: ['JobCategory', 'HighestEducationLevel', 'CareerLengthCZ', 'TeachingActivityDesc', 'ResearchActivityDesc', 'FundedResearchActivityDesc', 'AdminActivityDesc']

--- PASO 1: Filtrando filas con > 6 NaNs (no estructurales) ---
Filas antes: 2747
Filas después: 2416
Filas eliminadas en Paso 1: 331

--- PASO 2: Eliminando outliers de Ingresos (IQR) ---
  Límites IQR para IncomeEURO: Inferior=-1904.77, Superior=7062.86
  Outliers detectados en IncomeEURO: 138
  Límites IQR para EuroAdjusted: Inferior=-1153.97, Superior=6231.46
  Outliers detectados en EuroAdjusted: 143
Filas antes: 2416
Filas después: 2273
Filas eliminadas en Paso 2: 143

--- PASO 3: Imputando NaNs restantes con KNN (k=5) ---
Columnas objetivo cuya imputación se verificará explícitamente: ['Age', 'HeiActualWeeklyHours', 'EffortLevel', 'EffortPercentage', 'IncomeEURO', 'EuroAdjusted', 'SalaryEff

In [61]:
# --- Bloque 4: Limpieza Final por Subgrupos (Paso 5) ---

# Asumiendo que 'df_final_cleaned' es el DataFrame resultante del Bloque 3 (Pasos 1-4)
if 'df_final_cleaned' not in locals() or df_final_cleaned is None:
    print("ERROR: El DataFrame 'df_final_cleaned' (salida del Bloque 3) no está definido.")
    # exit() # O maneja el error
elif df_final_cleaned.empty:
    print("ERROR: El DataFrame 'df_final_cleaned' está vacío. No se puede continuar con la limpieza por subgrupos.")
    df_fully_cleaned = df_final_cleaned.copy() # Mantener el df vacío
else:
    print(f"\nIniciando Bloque 4 (Limpieza final por subgrupos) sobre df_final_cleaned con shape: {df_final_cleaned.shape}")
    df = df_final_cleaned.copy() # Trabajar sobre una copia

    print("\n--- PASO 5: Eliminando filas con NaNs restantes DENTRO de subgrupos ---")
    print(f"Total de filas ANTES de la eliminación por subgrupo: {len(df)}")

    # --- Definir columnas estructurales por grupo (NOMBRES FINALES CamelCase) ---
    # Estas son las columnas donde NO se permitirán NaNs para cada grupo específico
    cols_academic_structural = [
        'TeachingActivityDesc',     # Antes: 1. Teaching...
        'ResearchActivityDesc',     # Antes: 2. Research...
        'FundedResearchActivityDesc',# Antes: 3. Activities related...
        'AdminActivityDesc'         # Antes: 4. Organisational...
    ]
    cols_non_academic_structural = [
        'JobCategory',              # Antes: 16. Please choose...
        'HighestEducationLevel',    # Antes: 17. The highest level...
        'CareerLengthCZ'           # Antes: 18. Total length...
    ]

    # Verificar que estas columnas existen en df
    cols_academic_structural = [col for col in cols_academic_structural if col in df.columns]
    cols_non_academic_structural = [col for col in cols_non_academic_structural if col in df.columns]
    print(f"Columnas críticas para Académicos (NaNs no permitidos aquí): {cols_academic_structural}")
    print(f"Columnas críticas para No Académicos (NaNs no permitidos aquí): {cols_non_academic_structural}")

    # --- Filtrar y Limpiar Subgrupo Académico ---
    # USAR NOMBRE FINAL 'AcademicOrNonAcademic'
    # ¡¡VERIFICAR SI 2 = Académico es correcto para tus datos!!
    academic_filter_col = 'AcademicOrNonAcademic'
    academic_code = 2
    df_academic = pd.DataFrame(columns=df.columns) # Inicializar vacío

    if academic_filter_col in df.columns:
        df_academic_filtered = df[df[academic_filter_col] == academic_code].copy() # Usar .copy()
        rows_before_academic = len(df_academic_filtered)
        print(f"\n--- Subgrupo: Académicos ({academic_filter_col} == {academic_code}) ---")
        print(f"Filas antes de limpiar: {rows_before_academic}")

        if not df_academic_filtered.empty:
            if cols_academic_structural:
                print("NaNs a eliminar en columnas relevantes para Académicos:")
                print(df_academic_filtered[cols_academic_structural].isnull().sum())
                # Eliminar filas con NaNs en las columnas estructurales académicas
                df_academic = df_academic_filtered.dropna(subset=cols_academic_structural) # Guardar resultado en df_academic
                rows_after_academic = len(df_academic)
                print(f"Filas después de limpiar: {rows_after_academic}")
                print(f"Filas eliminadas en subgrupo Académico: {rows_before_academic - rows_after_academic}")
            else:
                print("Advertencia: No se encontraron columnas estructurales académicas para verificar NaNs.")
                df_academic = df_academic_filtered # Mantener sin cambios
        else:
             print("Subgrupo Académico está vacío.")
             df_academic = df_academic_filtered # Mantener vacío

    else:
        print(f"Advertencia: Columna '{academic_filter_col}' no encontrada. No se puede procesar subgrupo Académico.")


    # --- Filtrar y Limpiar Subgrupo No Académico ---
    # USAR NOMBRE FINAL 'AcademicOrNonAcademic'
    # ¡¡VERIFICAR SI 1 = No Académico es correcto para tus datos!!
    non_academic_code = 1
    df_non_academic = pd.DataFrame(columns=df.columns) # Inicializar vacío

    if academic_filter_col in df.columns:
        df_non_academic_filtered = df[df[academic_filter_col] == non_academic_code].copy() # Usar .copy()
        rows_before_non_academic = len(df_non_academic_filtered)
        print(f"\n--- Subgrupo: No Académicos ({academic_filter_col} == {non_academic_code}) ---")
        print(f"Filas antes de limpiar: {rows_before_non_academic}")

        if not df_non_academic_filtered.empty:
            if cols_non_academic_structural:
                print("NaNs a eliminar en columnas relevantes para No Académicos:")
                print(df_non_academic_filtered[cols_non_academic_structural].isnull().sum())
                # Eliminar filas con NaNs en las columnas estructurales no académicas
                df_non_academic = df_non_academic_filtered.dropna(subset=cols_non_academic_structural) # Guardar resultado en df_non_academic
                rows_after_non_academic = len(df_non_academic)
                print(f"Filas después de limpiar: {rows_after_non_academic}")
                print(f"Filas eliminadas en subgrupo No Académico: {rows_before_non_academic - rows_after_non_academic}")
            else:
                print("Advertencia: No se encontraron columnas estructurales no académicas para verificar NaNs.")
                df_non_academic = df_non_academic_filtered # Mantener sin cambios
        else:
            print("Subgrupo No Académico está vacío.")
            df_non_academic = df_non_academic_filtered # Mantener vacío
    else:
         print(f"Advertencia: Columna '{academic_filter_col}' no encontrada. No se puede procesar subgrupo No Académico.")


    # --- Combinar los DataFrames Limpios ---
    print("\nRecombinando los subgrupos limpios...")
    # Solo concatenar si los dataframes tienen datos
    if not df_academic.empty or not df_non_academic.empty:
        df_fully_cleaned = pd.concat([df_academic, df_non_academic], ignore_index=True)
        print(f"DataFrame recombinado tiene {df_fully_cleaned.shape[0]} filas.")
    else:
        print("Error: Ambos subgrupos (Académico y No Académico) están vacíos o no se pudieron procesar.")
        df_fully_cleaned = pd.DataFrame(columns=df.columns) # Resultado vacío

    # --- Verificación Final ---
    print("\nVerificación final de NaNs después de eliminar por subgrupo:")
    if not df_fully_cleaned.empty:
        final_nan_counts = df_fully_cleaned.isnull().sum()
        print("Conteo de NaNs por columna en el DataFrame final:")
        # Mostrar solo columnas que AÚN tengan NaNs (deberían ser las ignoradas en Paso 4 + estructurales no relevantes al subgrupo)
        print(final_nan_counts[final_nan_counts > 0])
    else:
        print("El DataFrame final está vacío.")


    # El DataFrame 'df_fully_cleaned' es el resultado final de todo el proceso.
    print("\n==============================================================================")
    print("PROCESO FINAL DE LIMPIEZA POR SUBGRUPOS COMPLETADO")
    if not df_fully_cleaned.empty:
        print(f"El DataFrame final 'df_fully_cleaned' tiene: {df_fully_cleaned.shape[0]} filas y {df_fully_cleaned.shape[1]} columnas.")
    else:
        print("El DataFrame final 'df_fully_cleaned' está vacío.")
    print("==============================================================================")


Iniciando Bloque 4 (Limpieza final por subgrupos) sobre df_final_cleaned con shape: (2273, 59)

--- PASO 5: Eliminando filas con NaNs restantes DENTRO de subgrupos ---
Total de filas ANTES de la eliminación por subgrupo: 2273
Columnas críticas para Académicos (NaNs no permitidos aquí): ['TeachingActivityDesc', 'ResearchActivityDesc', 'FundedResearchActivityDesc', 'AdminActivityDesc']
Columnas críticas para No Académicos (NaNs no permitidos aquí): ['JobCategory', 'HighestEducationLevel', 'CareerLengthCZ']

--- Subgrupo: Académicos (AcademicOrNonAcademic == 2) ---
Filas antes de limpiar: 1787
NaNs a eliminar en columnas relevantes para Académicos:
TeachingActivityDesc           27
ResearchActivityDesc           37
FundedResearchActivityDesc    104
AdminActivityDesc              38
dtype: int64
Filas después de limpiar: 1666
Filas eliminadas en subgrupo Académico: 121

--- Subgrupo: No Académicos (AcademicOrNonAcademic == 1) ---
Filas antes de limpiar: 486
NaNs a eliminar en columnas rel

In [66]:
# --- Bloque 5: Limpieza Adicional Post-Subgrupos (NUEVO PASO 6) ---

# Asegúrate de que 'df_fully_cleaned' exista y sea el resultado del Paso 5
if 'df_fully_cleaned' not in locals() or df_fully_cleaned is None:
    print("ERROR: El DataFrame 'df_fully_cleaned' (salida del Paso 5) no está definido.")
    # exit()
elif df_fully_cleaned.empty:
    print("ERROR: El DataFrame 'df_fully_cleaned' está vacío. No se puede aplicar limpieza adicional.")
    df_final_step6 = df_fully_cleaned.copy() # Mantener vacío
else:
    print("\n--- PASO 6: Eliminando filas con NaNs restantes en columnas específicas ---")
    # Trabajar sobre una copia del resultado del Paso 5
    df_final_step6 = df_fully_cleaned.copy()
    print(f"Filas ANTES del dropna final selectivo: {len(df_final_step6)}")

    # Definir columnas donde los NaNs son DEFINITIVAMENTE aceptables al final
    # Las 7 estructurales + SalaryPerHour por ser object y necesitar limpieza aparte
    structural_nan_cols_final = [ # Re-obtener nombres por si acaso
        'JobCategory', 'HighestEducationLevel', 'CareerLengthCZ',
        'TeachingActivityDesc', 'ResearchActivityDesc', 'FundedResearchActivityDesc',
        'AdminActivityDesc'
    ]
    cols_to_always_ignore_nans = structural_nan_cols_final + ['SalaryPerHour']
    # Asegurarse que existen en el df actual
    cols_to_always_ignore_nans = [col for col in cols_to_always_ignore_nans if col in df_final_step6.columns]

    # Definir las columnas a REVISAR en este dropna final: TODAS MENOS las ignoradas
    all_cols_step6 = df_final_step6.columns.tolist()
    cols_to_check_final = [col for col in all_cols_step6 if col not in cols_to_always_ignore_nans]

    print(f"Se REVISARÁN {len(cols_to_check_final)} columnas para el dropna final (ej: Gender, MaritalStatus...).")
    print(f"Se IGNORARÁN NaNs en: {cols_to_always_ignore_nans}")

    # Ver NaNs ANTES de dropear en las columnas a revisar
    nans_before_final_drop = df_final_step6[cols_to_check_final].isnull().sum()
    print("\nNaNs a eliminar en este paso (por columna):")
    print(nans_before_final_drop[nans_before_final_drop > 0])
    total_nans_rows_affected = len(df_final_step6[df_final_step6[cols_to_check_final].isnull().any(axis=1)])
    print(f"\nNúmero de filas con al menos un NaN en columnas a revisar: {total_nans_rows_affected}")


    # Aplicar dropna si hay columnas para revisar y NaNs presentes
    if cols_to_check_final and total_nans_rows_affected > 0 :
        df_final_step6.dropna(subset=cols_to_check_final, inplace=True)
    else:
        print("No se encontraron NaNs en las columnas a revisar o no hay columnas para revisar.")


    rows_after_step6 = len(df_final_step6)
    print(f"\nFilas ANTES de este paso: {len(df_fully_cleaned)}") # Comparar con la entrada
    print(f"Filas DESPUÉS de este paso: {rows_after_step6}")
    print(f"Filas eliminadas en Paso 6: {len(df_fully_cleaned) - rows_after_step6}")

    # Verificación final de NaNs
    print("\nNaNs restantes finales:")
    final_nan_summary = df_final_step6.isnull().sum()
    # Mostrar solo las columnas que aún tengan NaNs (deberían ser solo las ignoradas)
    print(final_nan_summary[final_nan_summary > 0])

    print("\n==============================================================================")
    print("PROCESO COMPLETO (INCLUYENDO PASO 6) FINALIZADO")
    print(f"El DataFrame final tiene: {df_final_step6.shape[0]} filas y {df_final_step6.shape[1]} columnas.")
    print("==============================================================================")

# Guardar el resultado final si lo deseas
# final_output_file = 'datos_finales_limpios.xlsx'
# df_final_step6.to_excel(final_output_file, index=False)
# print(f"DataFrame final guardado en: {final_output_file}")


--- PASO 6: Eliminando filas con NaNs restantes en columnas específicas ---
Filas ANTES del dropna final selectivo: 2095
Se REVISARÁN 51 columnas para el dropna final (ej: Gender, MaritalStatus...).
Se IGNORARÁN NaNs en: ['JobCategory', 'HighestEducationLevel', 'CareerLengthCZ', 'TeachingActivityDesc', 'ResearchActivityDesc', 'FundedResearchActivityDesc', 'AdminActivityDesc', 'SalaryPerHour']

NaNs a eliminar en este paso (por columna):
Gender                         4
MaritalStatus                 15
CareResponsibilities          12
HeiType                        1
FacultySubjectArea            20
EmploymentContractDuration     2
LeadershipPosition             8
PolicyInfluence                4
dtype: int64

Número de filas con al menos un NaN en columnas a revisar: 62

Filas ANTES de este paso: 2095
Filas DESPUÉS de este paso: 2033
Filas eliminadas en Paso 6: 62

NaNs restantes finales:
SalaryPerHour                  124
JobCategory                   1626
HighestEducationLevel      

In [67]:
# --- PASO 7: Eliminando filas con NaN en SalaryPerHour ---

# Asumiendo que df_final_step6 es el resultado del Paso 6 (2033 filas)
if 'df_final_step6' not in locals() or df_final_step6 is None:
    print("ERROR: El DataFrame 'df_final_step6' (salida del Paso 6) no está definido.")
    # exit()
elif df_final_step6.empty:
    print("ERROR: El DataFrame 'df_final_step6' está vacío.")
    df_final_definitivo = df_final_step6.copy() # Mantener vacío
else:
    print("\n--- PASO 7: Eliminando filas con NaN específicamente en SalaryPerHour ---")
    # Trabajar sobre una copia por seguridad
    df_final_definitivo = df_final_step6.copy()
    print(f"Filas ANTES de eliminar NaNs de SalaryPerHour: {len(df_final_definitivo)}")

    target_col = 'SalaryPerHour'
    if target_col in df_final_definitivo.columns:
        nans_before_drop = df_final_definitivo[target_col].isnull().sum()
        print(f"NaNs encontrados en '{target_col}' antes de drop: {nans_before_drop}")

        # Eliminar filas donde la columna específica 'SalaryPerHour' es NaN
        df_final_definitivo.dropna(subset=[target_col], inplace=True)

        rows_after_drop = len(df_final_definitivo)
        print(f"Filas DESPUÉS de eliminar NaNs de SalaryPerHour: {rows_after_drop}")
        # Comparar con la entrada a ESTE paso
        print(f"Filas eliminadas en este paso: {len(df_final_step6) - rows_after_drop}")

        # Verificación final de NaNs
        print("\nNaNs restantes finales:")
        final_nan_summary = df_final_definitivo.isnull().sum()
        # Mostrar solo las columnas que AÚN tengan NaNs (deberían ser solo las estructurales)
        print(final_nan_summary[final_nan_summary > 0])

        print("\n==============================================================================")
        print("PROCESO COMPLETO (INCLUYENDO PASO 7) FINALIZADO")
        print(f"El DataFrame FINAL DEFINITIVO tiene: {df_final_definitivo.shape[0]} filas y {df_final_definitivo.shape[1]} columnas.")
        print("==============================================================================")

        # Guardar el resultado final si lo deseas
        # final_output_file_def = 'datos_finales_definitivos.xlsx'
        # df_final_definitivo.to_excel(final_output_file_def, index=False)
        # print(f"DataFrame final guardado en: {final_output_file_def}")

    else:
        print(f"ERROR: Columna '{target_col}' no encontrada. No se eliminaron filas.")
        df_final_definitivo = df_final_step6.copy() # Mantener el df anterior

# Ahora 'df_final_definitivo' contiene el resultado


--- PASO 7: Eliminando filas con NaN específicamente en SalaryPerHour ---
Filas ANTES de eliminar NaNs de SalaryPerHour: 2033
NaNs encontrados en 'SalaryPerHour' antes de drop: 124
Filas DESPUÉS de eliminar NaNs de SalaryPerHour: 1909
Filas eliminadas en este paso: 124

NaNs restantes finales:
JobCategory                   1502
HighestEducationLevel         1502
CareerLengthCZ                1502
TeachingActivityDesc           407
ResearchActivityDesc           407
FundedResearchActivityDesc     407
AdminActivityDesc              407
dtype: int64

PROCESO COMPLETO (INCLUYENDO PASO 7) FINALIZADO
El DataFrame FINAL DEFINITIVO tiene: 1909 filas y 59 columnas.


In [68]:
# --- PASO FINAL: Guardar el DataFrame Limpio ---

# Asegúrate de que el DataFrame final se llame 'df_final_definitivo'
# o cambia el nombre de la variable si usaste otro en el último paso.
final_dataframe_to_save = df_final_definitivo # Cambia 'df_final_definitivo' si es necesario

if 'final_dataframe_to_save' in locals() and not final_dataframe_to_save.empty:

    # Define los nombres de archivo en inglés
    csv_filename = 'final_cleaned_data.csv'
    excel_filename = 'final_cleaned_data.xlsx'

    print(f"\n--- Guardando el DataFrame final ({final_dataframe_to_save.shape[0]} filas, {final_dataframe_to_save.shape[1]} columnas) ---")

    # Guardar en archivo CSV
    try:
        print(f"Guardando en formato CSV: {csv_filename} ...")
        # index=False evita que se escriba el índice del DataFrame como una columna en el archivo
        # encoding='utf-8' es una codificación estándar que maneja bien diversos caracteres
        final_dataframe_to_save.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"Archivo CSV '{csv_filename}' guardado exitosamente.")
    except Exception as e:
        print(f"ERROR al guardar el archivo CSV: {e}")

    # Guardar en archivo Excel
    try:
        print(f"Guardando en formato Excel: {excel_filename} ...")
        # index=False tiene el mismo propósito que en to_csv
        # sheet_name permite nombrar la hoja dentro del archivo Excel
        final_dataframe_to_save.to_excel(excel_filename, index=False, sheet_name='CleanedData')
        print(f"Archivo Excel '{excel_filename}' guardado exitosamente.")
    except Exception as e:
        print(f"ERROR al guardar el archivo Excel: {e}")

else:
    print("\nERROR: No se encontró el DataFrame final o está vacío. No se guardaron archivos.")


--- Guardando el DataFrame final (1909 filas, 59 columnas) ---
Guardando en formato CSV: final_cleaned_data.csv ...
Archivo CSV 'final_cleaned_data.csv' guardado exitosamente.
Guardando en formato Excel: final_cleaned_data.xlsx ...
Archivo Excel 'final_cleaned_data.xlsx' guardado exitosamente.


In [69]:
import pandas as pd
import numpy as np

# --- PASO ADICIONAL: Generar Información para Diccionario de Datos ---

# Asegúrate de que 'df_final_definitivo' contiene tu DataFrame final limpio
if 'df_final_definitivo' in locals() and not df_final_definitivo.empty:
    print("\n" + "="*70)
    print("--- Información para el Diccionario de Datos ---")
    print(f"Dataset: df_final_definitivo | Filas: {df_final_definitivo.shape[0]}, Columnas: {df_final_definitivo.shape[1]}")
    print("="*70 + "\n")

    df_to_describe = df_final_definitivo

    for col in df_to_describe.columns:
        print(f"--- Columna: '{col}' ---")

        # 1. Tipo de Dato
        dtype = df_to_describe[col].dtype
        print(f"  Tipo de Dato (dtype): {dtype}")

        # 2. Número de Valores No Nulos
        non_null_count = df_to_describe[col].notnull().sum()
        print(f"  Valores No Nulos: {non_null_count} / {len(df_to_describe)}") # Muestra sobre el total de filas

        # 3. Número de Valores Nulos (NaN)
        null_count = df_to_describe[col].isnull().sum()
        if null_count > 0:
            print(f"  Valores Nulos (NaN): {null_count} ({null_count / len(df_to_describe):.2%})") # Muestra también porcentaje
        else:
            print(f"  Valores Nulos (NaN): 0")

        # 4. Número de Valores Únicos
        unique_count = df_to_describe[col].nunique()
        print(f"  Número de Valores Únicos: {unique_count}")

        # 5. Ejemplos de Valores Únicos (si son pocos)
        # Útil para entender los niveles de variables categóricas o códigos
        if unique_count > 0 and unique_count <= 25: # Puedes ajustar este umbral (e.g., 15, 20, 30)
            try:
                # Intenta obtener y ordenar los valores únicos (sin NaNs)
                unique_vals = sorted(list(df_to_describe[col].dropna().unique()))
                print(f"  Valores Únicos Presentes: {unique_vals}")
            except TypeError: # Falla si hay tipos mixtos que no se pueden ordenar
                unique_vals = list(df_to_describe[col].dropna().unique())
                print(f"  Valores Únicos Presentes (sin ordenar): {unique_vals}")
        elif unique_count > 0:
            # Si hay muchos valores únicos, muestra algunos ejemplos
            examples = df_to_describe[col].dropna().unique()[:5] # Muestra hasta 5 ejemplos
            print(f"  Ejemplos de Valores (primeros 5 únicos): {list(examples)}")


        # 6. Estadísticas Básicas (para columnas numéricas)
        # Usamos is_numeric_dtype para incluir int y float
        if pd.api.types.is_numeric_dtype(dtype):
            min_val = df_to_describe[col].min()
            max_val = df_to_describe[col].max()
            mean_val = df_to_describe[col].mean()
            median_val = df_to_describe[col].median()
            std_val = df_to_describe[col].std()
            print(f"  Estadísticas (Numéricas):")
            print(f"    Mín: {min_val:.2f}")
            print(f"    Máx: {max_val:.2f}")
            print(f"    Media: {mean_val:.2f}")
            print(f"    Mediana: {median_val:.2f}")
            print(f"    Desv. Estándar: {std_val:.2f}")
        # Caso especial para SalaryPerHour que sabemos que es object pero debería ser numérico
        elif col == 'SalaryPerHour' and dtype == 'object':
             print("  Estadísticas: (Columna tipo 'object', requiere limpieza para análisis numérico)")

        print("-" * 40) # Separador entre columnas

else:
    print("\nERROR: No se encontró el DataFrame final 'df_final_definitivo' o está vacío.")


--- Información para el Diccionario de Datos ---
Dataset: df_final_definitivo | Filas: 1909, Columnas: 59

--- Columna: 'Country' ---
  Tipo de Dato (dtype): int64
  Valores No Nulos: 1909 / 1909
  Valores Nulos (NaN): 0
  Número de Valores Únicos: 2
  Valores Únicos Presentes: [np.int64(1), np.int64(2)]
  Estadísticas (Numéricas):
    Mín: 1.00
    Máx: 2.00
    Media: 1.59
    Mediana: 2.00
    Desv. Estándar: 0.49
----------------------------------------
--- Columna: 'Version' ---
  Tipo de Dato (dtype): int64
  Valores No Nulos: 1909 / 1909
  Valores Nulos (NaN): 0
  Número de Valores Únicos: 4
  Valores Únicos Presentes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Estadísticas (Numéricas):
    Mín: 1.00
    Máx: 4.00
    Media: 1.70
    Mediana: 2.00
    Desv. Estándar: 0.63
----------------------------------------
--- Columna: 'Gender' ---
  Tipo de Dato (dtype): float64
  Valores No Nulos: 1909 / 1909
  Valores Nulos (NaN): 0
  Número de Valores Únicos: 3
  Valores Ú